#### Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

- How to create test datasets
- How to run your RAG application on those datasets
- How to measure your application's performance using different evaluation metrics

Overview
A typical RAG evaluation workflow consists of three main steps:
- Creating a dataset with questions and their expected answers
- Running your RAG application on those questions
- Using evaluators to measure how well your application performed, looking at factors like:
  - Answer relevance
  - Answer accuracy
  - Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

#### Chatbot Evaluation

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

In [ ]:
from langsmith import Client

client = Client()

# define dataset
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

#### Define Metrics - LLM as a Judge

In [6]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=user_content,
        config=types.GenerateContentConfig(
            system_instruction=eval_instructions,
            temperature=0,
            max_output_tokens=10
        )
    )
    
    result = response.text.strip().upper()
    return result == "CORRECT"

In [14]:
## Concisions- checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs: dict) -> bool:
    predicted_answer = outputs["response"]
    reference_answer = reference_outputs["answer"]
    return len(predicted_answer) < 2 * len(reference_answer)

#### Run Evaluations

In [15]:
default_instructions = (
    "Respond to the user's question in a short, concise manner "
    "(one short sentence)."
)
def my_app(question: str, model: str="gemini-2.5-flash-lite", instructions: str=default_instructions) -> str:
    response = gemini_client.models.generate_content(
        model=model, 
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0
        )
    )
    return response.text.strip()

In [16]:
### Call my_app for every datapoints
def ls_target(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="gemini-2.5-flash-lite",
        )
    }

In [17]:
# run our application
experiment_result = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-2.5-flash-lite-chatbot"
)

View the evaluation results for experiment: 'gemini-2.5-flash-lite-chatbot-e23826a4' at:
https://smith.langchain.com/o/7bc34ce6-22a4-465f-b498-a344507aba68/datasets/5994a8ac-7398-4edb-9575-e96bb3dff592/compare?selectedSessions=af6880b1-d518-4607-9627-64b36803d0e3




5it [00:10,  2.04s/it]


In [ ]:
# Call my_app for every data point
def ls_target_gemini_flash(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="gemini-2.5-flash",
        )
    }


# Run the evaluation
experiment_results = client.evaluate(
    ls_target_gemini_flash,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-2.5-flash-chatbot",
)

#### Evaluation for RAG

In [11]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
# NOTE: to handle model rate limiting will use only one doc for now
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    # "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    # "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# load docs from URL
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=250, chunk_overlap=0)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)
print(len(doc_splits))

# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(documents=doc_splits, embedding=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
))

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=3)

62


In [12]:
retriever.invoke("what is agents")

[Document(id='aeb46dc3-0b12-4661-ae6c-8068165a84a2', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [13]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-google-genai': '4.3.2'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash-lite', client=<google.genai.client.Client object at 0x00000214CAA63490>, default_metadata=(), model_kwargs={})

In [14]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [15]:
rag_bot("What is agents")

{'answer': "Agents are systems where a large language model (LLM) acts as the core controller, functioning like the agent's brain. They are complemented by components like planning, memory, and tool use to achieve goals. These agents can break down complex tasks into smaller subgoals and refine their actions through self-reflection.",
 'documents': [Document(id='aeb46dc3-0b12-4661-ae6c-8068165a84a2', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain,

#### Dataset

In [16]:
from langsmith import Client

client = Client()

# Define the examples for the dataset
# NOTE: to handle model rate limiting will use only one example related to the chunked doc for now
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    # {
    #     "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
    #     "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    # },
    # {
    #     "inputs": {"question": "What are five types of adversarial attacks?"},
    #     "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    # }
]

### create the daatset and example in LAngsmith
dataset_name = "RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['de4fede1-5625-423b-8d6f-b60fed1dbf36'],
 'count': 1,
 'as_of': '2026-09-11T14:14:06.484032652Z'}

#### Evaluators or Metrics
1. Correctness: Response vs reference answer
    - Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
    - Mode: Requires a ground truth (reference) answer supplied through a dataset
    - Evaluator: Use LLM-as-judge to assess answer correctness.

In [17]:
from typing_extensions import Annotated, TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI


# Correctness output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score",
    ]
    correct: Annotated[
        bool,
        ...,
        "True if the answer is correct, False otherwise.",
    ]


# Correctness prompt
correctness_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER,
and the STUDENT ANSWER.

Here is the grade criteria to follow:

1. Grade the student's answer based only on its factual accuracy
   relative to the ground truth answer.
2. Ensure that the student answer does not contain any conflicting statements.
3. It is acceptable if the student answer contains more information
   than the ground truth answer, as long as it is factually accurate
   relative to the ground truth answer.

Correctness:

A correctness value of True means that the student's answer meets
all of the criteria.

A correctness value of False means that the student's answer does
not meet all of the criteria.

Explain your reasoning step by step before giving the final conclusion.
Avoid simply stating the correct answer at the outset.
"""


# LangChain Google GenAI model
grader_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
)


# Apply structured output
structured_grader = grader_llm.with_structured_output(
    CorrectnessGrade,
    method="json_schema",
)


# Evaluator
def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> bool:
    """An evaluator for RAG answer accuracy."""

    answers = f"""
QUESTION:
{inputs["question"]}

GROUND TRUTH ANSWER:
{reference_outputs["answer"]}

STUDENT ANSWER:
{outputs["answer"]}
"""

    grade = structured_grader.invoke(
        [
            {
                "role": "system",
                "content": correctness_instructions,
            },
            {
                "role": "user",
                "content": answers,
            },
        ]
    )

    return grade["correct"]

#### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [18]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[
        str,
        ...,
        "Explain your reasoning for the score",
    ]
    relevant: Annotated[
        bool,
        ...,
        "Provide the score on whether the answer addresses the question",
    ]


# Grade prompt
relevance_instructions = """
You are a teacher grading a quiz.

You will be given a QUESTION and a STUDENT ANSWER.

Follow these grading criteria:

1. Ensure that the STUDENT ANSWER is concise and relevant to the QUESTION.
2. Ensure that the STUDENT ANSWER helps answer the QUESTION.

Relevance:

A relevance value of True means that the student's answer meets
all of the criteria.

A relevance value of False means that the student's answer does not
meet all of the criteria.

Explain your reasoning step by step to ensure your reasoning and
conclusion are correct.

Avoid simply stating the correct answer at the outset.
"""


# Grader LLM
relevance_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
).with_structured_output(
    RelevanceGrade,
    method="json_schema",
)


# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""

    answer = (
        f"QUESTION: {inputs['question']}\n"
        f"STUDENT ANSWER: {outputs['answer']}"
    )

    grade = relevance_llm.invoke(
        [
            {
                "role": "system",
                "content": relevance_instructions,
            },
            {
                "role": "user",
                "content": answer,
            },
        ]
    )

    return grade["relevant"]

#### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [19]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
).with_structured_output(
    GroundedGrade,
    method="json_schema",
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

#### Retrieval Relevance: Retrieved docs vs input

In [23]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
).with_structured_output(
    RetrievalRelevanceGrade,
    method="json_schema",
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

#### Run the evaluation

In [24]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gemini-2.5-flash-lite-preview"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()

View the evaluation results for experiment: 'rag-doc-relevance-725c18b0' at:
https://smith.langchain.com/o/7bc34ce6-22a4-465f-b498-a344507aba68/datasets/fc88ee81-871e-4ca1-8746-25abf7aef204/compare?selectedSessions=e0f2b691-f412-4829-9674-abb0cc043a4d




1it [00:36, 36.31s/it]


,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,How does the ReAct agent use self-reflection?,The provided documents do not explicitly state...,[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",True,True,True,False,4.289231,de4fede1-5625-423b-8d6f-b60fed1dbf36,01a090d8-92e4-79c3-9996-a2bdf2c5204d
